In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")


if not openrouter_api_key:
    raise RuntimeError(
        "No OPENROUTER_API_KEY found. Add OPENROUTER_API_KEY=or_... to .env and save it."
    )
    
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

In [3]:
links = fetch_website_links("https://www.sc.com/en/")
links

['#content',
 'https://www.sc.com/en/',
 'https://www.sc.com/en/country-popup/online/',
 'https://pvm.standardchartered.com/pvb/index.html#/login',
 'https://s2b.standardchartered.com/',
 '/en/contact-us/',
 'https://www.sc.com/en/our-locations/',
 'https://www.sc.com/en/about/',
 'https://www.sc.com/en/investors/financial-results/annual-report/',
 'https://www.sc.com/en/about/who-we-are/',
 'https://www.sc.com/en/about/who-we-are/',
 'https://www.sc.com/en/about/',
 'https://www.sc.com/en/about/our-strategy/',
 'https://www.sc.com/en/about/our-business-model/',
 'https://www.sc.com/en/about/our-people/',
 'https://www.sc.com/en/our-locations/',
 'https://www.sc.com/en/country-popup/online/',
 'https://pvm.standardchartered.com/pvb/index.html#/login',
 'https://s2b.standardchartered.com/',
 '/en/contact-us/',
 'https://www.sc.com/en/our-locations/',
 'https://www.sc.com/en/about/who-we-are/',
 'https://www.sc.com/en/about/who-we-are/',
 'https://www.sc.com/en/about/who-we-are/',
 'http

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://www.sc.com/en/"))


Here is the list of links on the website https://www.sc.com/en/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#content
https://www.sc.com/en/
https://www.sc.com/en/country-popup/online/
https://pvm.standardchartered.com/pvb/index.html#/login
https://s2b.standardchartered.com/
/en/contact-us/
https://www.sc.com/en/our-locations/
https://www.sc.com/en/about/
https://www.sc.com/en/investors/financial-results/annual-report/
https://www.sc.com/en/about/who-we-are/
https://www.sc.com/en/about/who-we-are/
https://www.sc.com/en/about/
https://www.sc.com/en/about/our-strategy/
https://www.sc.com/en/about/our-business-model/
https://www.sc.com/en/about/our-people/
https://www.sc.com/en/our-locations/
https://www.sc.com/en/country-popup/online/
https://pvm.standardchartered.com/pvb/index.html#/login
https://s2b.s

In [7]:
def select_relevant_links(url):
    response = openrouter.chat.completions.create(
        model="nvidia/nemotron-3-nano-30b-a3b:free",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://www.sc.com/en/")

{'links': [{'type': 'about page', 'url': 'https://www.sc.com/en/about/'},
  {'type': 'our strategy', 'url': 'https://www.sc.com/en/about/our-strategy/'},
  {'type': 'our business model',
   'url': 'https://www.sc.com/en/about/our-business-model/'},
  {'type': 'our people', 'url': 'https://www.sc.com/en/about/our-people/'},
  {'type': 'sustainability page',
   'url': 'https://www.sc.com/en/about/sustainability/'},
  {'type': 'responsible business practices',
   'url': 'https://www.sc.com/en/about/sustainability/responsible-business-practices/our-net-zero-roadmap/'},
  {'type': 'investors page', 'url': 'https://www.sc.com/en/investors/'},
  {'type': 'annual report',
   'url': 'https://www.sc.com/en/investors/financial-results/annual-report/'},
  {'type': 'our locations', 'url': 'https://www.sc.com/en/our-locations/'}]}

In [9]:
MODEL="nvidia/nemotron-3-nano-30b-a3b:free"
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openrouter.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://www.sc.com/en/")

Selecting relevant links for https://www.sc.com/en/ by calling nvidia/nemotron-3-nano-30b-a3b:free
Found 12 relevant links


{'links': [{'type': 'about page', 'url': 'https://www.sc.com/en/about/'},
  {'type': 'who we are', 'url': 'https://www.sc.com/en/about/who-we-are/'},
  {'type': 'our strategy', 'url': 'https://www.sc.com/en/about/our-strategy/'},
  {'type': 'our business model',
   'url': 'https://www.sc.com/en/about/our-business-model/'},
  {'type': 'sustainability overview',
   'url': 'https://www.sc.com/en/about/sustainability/'},
  {'type': 'net zero roadmap',
   'url': 'https://www.sc.com/en/about/sustainability/responsible-business-practices/our-net-zero-roadmap/'},
  {'type': 'annual report',
   'url': 'https://www.sc.com/en/investors/financial-results/annual-report/'},
  {'type': 'global careers early careers',
   'url': 'https://www.sc.com/en/global-careers/early-careers/'},
  {'type': 'global careers experienced hire',
   'url': 'https://www.sc.com/en/global-careers/experienced-hire/'},
  {'type': 'global careers business areas',
   'url': 'https://www.sc.com/en/global-careers/business-areas/

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://www.sc.com/en/"))

Selecting relevant links for https://www.sc.com/en/ by calling nvidia/nemotron-3-nano-30b-a3b:free
Found 9 relevant links
## Landing Page:

Standard Chartered | Wealth, Corporate & Investment Banking

Skip to content
Standard Chartered
Online banking
Private banking
Straight2Bank
Contact us
Locations
About us
Home
Full Year Results 2025
We’ve released our full-year results and annual report.
See results
Who we are
Home
Who we are
Learn more
About Standard Chartered
Our strategy
Our business model
Our leadership and people
Our locations
Online banking
Private banking
Straight2Bank
Contact us
Locations
Our purpose and culture
Home
Our purpose and culture
Learn more
Our purpose
Our culture
Our sponsorships
Investing in communities
Online banking
Private banking
Straight2Bank
Contact us
Locations
Sustainability
Home
Sustainability
Learn more
Our approach
Our net zero roadmap
Position statements
Disclosures and frameworks
Online banking
Private banking
Straight2Bank
Contact us
Locations
Onl

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("Standard Chartered", "https://www.sc.com/en/")

Selecting relevant links for https://www.sc.com/en/ by calling nvidia/nemotron-3-nano-30b-a3b:free
Found 8 relevant links


'\nYou are looking at a company called: Standard Chartered\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nStandard Chartered | Wealth, Corporate & Investment Banking\n\nSkip to content\nStandard Chartered\nOnline banking\nPrivate banking\nStraight2Bank\nContact us\nLocations\nAbout us\nHome\nFull Year Results 2025\nWe’ve released our full-year results and annual report.\nSee results\nWho we are\nHome\nWho we are\nLearn more\nAbout Standard Chartered\nOur strategy\nOur business model\nOur leadership and people\nOur locations\nOnline banking\nPrivate banking\nStraight2Bank\nContact us\nLocations\nOur purpose and culture\nHome\nOur purpose and culture\nLearn more\nOur purpose\nOur culture\nOur sponsorships\nInvesting in communities\nOnline banking\nPrivate banking\nStraight2Bank\nContact us\nLocations\nSustainability\nHome\nSustainability\nLearn mo

In [16]:
def create_brochure(company_name, url):
    response = openrouter.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("Standard Chartered", "https://www.sc.com/en/")

Selecting relevant links for https://www.sc.com/en/ by calling nvidia/nemotron-3-nano-30b-a3b:free
Found 8 relevant links


# Standard Chartered – A Global Bank with Purpose  

## Who We Are  
Standard Chartered is a **leading international bank** that combines **cross‑border expertise** with deep wealth‑management insight.  Operating in more than **70 countries**, we serve **affluent individuals, families, SMEs, and corporates** who demand a blend of financial expertise and localized service.  

Our **purpose** is to “**be the bank that powers progress**” – helping people, businesses, and communities achieve their ambitions while driving responsible, sustainable growth.  

---

## Our Business Model & Key Segments  

| Segment | What It Offers | Typical Clients |
|---------|----------------|-----------------|
| **Wealth & Retail Banking** | Private, Priority & Personal Banking; SME solutions; wealth management tools | High‑net‑worth individuals, affluent families, small‑to‑medium enterprises |
| **Corporate & Investment Banking** | Transaction Banking, Global Markets, Global Banking, Islamic Banking | Multinational corporations, governments, large‑scale investors |
| **Investors & Shareholder Relations** | Transparent reporting, investor events, market updates | Institutional investors, analysts, shareholders |
| **Ventures & Innovation** | Digital banking propositions (e.g., Straight2Bank) and strategic partnerships | Tech‑savvy clients, fintech collaborators |

---

## Our Culture  

- **Purpose‑Driven:** Every action is tied to our commitments to **community investment, sustainability, and inclusion**.  
- **Collaborative & Inclusive:** A global, multicultural workforce where diverse perspectives shape decisions.  
- **Customer‑Centric Innovation:** Continuous investment in digital platforms (e.g., Straight2Bank) to deliver seamless, personalized banking experiences.  
- **Integrity & Compliance:** Strict adherence to regulatory standards and ethical conduct across all markets.  

---

## Sustainability & Community Impact  

- **Net‑Zero Roadmap:** Ambitious targets to achieve **net‑zero emissions by 2050**, backed by measurable milestones.  
- **Investing in Communities:** Initiatives such as **education scholarships, financial literacy programs, and local development projects** in the regions where we operate.  
- **Responsible Banking:** Position statements and frameworks that embed **environmental, social, and governance (ESG)** considerations into every client offering.  

---

## Careers & Opportunities  

### For Graduates & Early‑Career Professionals  
- **Graduate Rotations:** Exposure to multiple divisions (e.g., Wealth Management, Corporate Banking, Sustainability).  
- **Leadership Development Programs:** Structured mentorship, networking, and skill‑building workshops.  

### For Experienced Professionals  
- **Specialist Roles** in **Wealth & Retail Banking, Transaction Banking, Global Markets, Islamic Banking** and more.  
- **Digital Innovation Teams** focusing on fintech, mobile banking (Straight2Bank), and data analytics.  

### Why Join Us?  
- **Global Reach with Local Insight:** Work across continents while understanding regional markets.  
- **Purpose‑Led Environment:** Contribute to tangible social and environmental outcomes.  
- **Career Mobility:** Internal mobility programs allow staff to move across geographies and functions.  
- **Inclusive Workplace:** Robust diversity, equity, and inclusion policies foster a supportive environment for all.  

---

## Our Customers  

- **Affluent Individuals & Families:** Tailored wealth solutions, premium private banking, and bespoke investment strategies.  
- **SMEs & Entrepreneurs:** Dedicated banking packages, financing options, and advisory services.  
- **Multinational Corporations:** End‑to‑end transaction services, trade finance, and market‑linked capital solutions.  
- **Institutional Investors:** Access to market insights, performance reporting, and investment opportunities.  

---

## Investor Highlights  

- **Robust Financial Performance:** Consistent delivery of **strong returns, capital adequacy, and risk management** as demonstrated in the latest Full‑Year Results 2025.  
- **Transparent Reporting:** Regular updates via the **Investor Hub**, including earnings releases, ESG disclosures, and market commentary.  
- **Strategic Growth:** Expansion of **digital banking platforms** and **wealth‑management capabilities** to capture emerging market opportunities.  

---

## Get in Touch  

- **Online Banking & Private Banking:** Explore seamless digital channels and personalized service options.  
- **Contact Us:** Connect with regional offices through the **Locations** page or schedule a meeting with a relationship manager.  
- **Investor Relations:** Access the latest financial data, presentations, and conference call schedules on the **Investors Hub**.  

---  

Standard Chartered blends **global reach, purpose‑driven values, and innovative financial solutions** to empower clients, investors, and talent alike. Whether you’re seeking a banking partner, an investment opportunity, or a rewarding career, we invite you to join us on the journey toward sustainable progress.